# 🚀 Walkability Semantic Segmentation: Google Colab Training

This notebook runs the complete 4-class semantic segmentation training pipeline for **BiSeNetV2** using Google Colab's free **Tesla T4 GPU (15-16 GB VRAM)**.

### ⚠️ Step 0: Ensure GPU Acceleration is Enabled
1. Go to the top menu: **Runtime** -> **Change runtime type**
2. Under *Hardware accelerator*, select **T4 GPU**
3. Click **Save**

## 1. Verify GPU Availability

In [1]:
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

Mon Oct 26 04:47:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive
Mount your Google Drive to access the dataset zip files and to automatically persist trained checkpoints.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Clone Repository & Checkout Training Branch

In [3]:
import os
%cd /content

if not os.path.exists("/content/Capstone-Project"):
    !git clone https://github.com/dev-june05/Capstone-Project.git

%cd /content/Capstone-Project
!git checkout divyansh/model-training
!git pull origin divyansh/model-training

/content
Cloning into 'Capstone-Project'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 70 (delta 19), reused 60 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 201.63 KiB | 1.13 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/Capstone-Project
branch 'divyansh/model-training' set up to track 'origin/divyansh/model-training'.
Switched to a new branch 'divyansh/model-training'
From https://github.com/dev-june05/Capstone-Project
 * branch            divyansh/model-training -> FETCH_HEAD
Already up to date.


## 4. Install Dependencies

In [4]:
!pip install -r requirements.txt
!pip install albumentations

INFO: pip is looking at multiple versions of gtts to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of wandb to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of wandb to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 105.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 75.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 73.4 MB/s  0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.5.0
    Uninstalling click-8.5.0:
      Successfully uninstalled click-8.5.0
  Attempting uninstall: wandb
    Found existing installation: wandb 0.28.1
    Uninstalling wandb-0.28.1:
      Succe

## 5. Unzip Cityscapes from Google Drive to Local Colab SSD
> **Tip:** Unzipping directly onto Colab's local NVMe SSD (`/content/Capstone-Project/data/cityscapes/`) takes only 1-2 minutes and makes training much faster than reading directly from Google Drive over network I/O.

Upload `leftImg8bit_trainvaltest.zip` and `gtFine_trainvaltest.zip` to your Google Drive (e.g. inside a folder named `Cityscapes`).

In [5]:
import os
from pathlib import Path

# Ensure target directory exists
dest_dir = "/content/Capstone-Project/data/cityscapes"
os.makedirs(dest_dir, exist_ok=True)

# Set this to the folder in Google Drive where you placed the zip files
DRIVE_DATASET_DIR = "/content/drive/MyDrive/Cityscapes"

leftimg_zip = os.path.join(DRIVE_DATASET_DIR, "leftImg8bit_trainvaltest.zip")
gtfine_zip = os.path.join(DRIVE_DATASET_DIR, "gtFine_trainvaltest.zip")

if os.path.exists(leftimg_zip):
    print("Unzipping leftImg8bit images...")
    !unzip -q "{leftimg_zip}" -d "{dest_dir}"
    print("Images unzipped successfully!")
else:
    print(f"❌ Not found: {leftimg_zip}. Please check your Google Drive folder name.")

if os.path.exists(gtfine_zip):
    print("Unzipping gtFine masks...")
    !unzip -q "{gtfine_zip}" -d "{dest_dir}"
    print("Masks unzipped successfully!")
else:
    print(f"❌ Not found: {gtfine_zip}. Please check your Google Drive folder name.")

Unzipping leftImg8bit images...
Images unzipped successfully!
Unzipping gtFine masks...
replace /content/Capstone-Project/data/cityscapes/README? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/Capstone-Project/data/cityscapes/license.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y

y
Masks unzipped successfully!


## 6. Verify Dataset Loading

In [6]:
# Verify that CityscapesDataset can detect the images and masks
from utils.dataset import CityscapesDataset

train_ds = CityscapesDataset("data/cityscapes", split="train")
val_ds = CityscapesDataset("data/cityscapes", split="val")
print(f"✅ Training samples detected: {len(train_ds)} (expected: 2975)")
print(f"✅ Validation samples detected: {len(val_ds)} (expected: 500)")

✅ Training samples detected: 2975 (expected: 2975)
✅ Validation samples detected: 500 (expected: 500)


In [10]:
%cd /content/Capstone-Project
!git pull origin divyansh/model-training

/content/Capstone-Project
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 656 bytes | 328.00 KiB/s, done.
From https://github.com/dev-june05/Capstone-Project
 * branch            divyansh/model-training -> FETCH_HEAD
   e065f4c..0049757  divyansh/model-training -> origin/divyansh/model-training
Updating e065f4c..0049757
Fast-forward
 models/bisenetv2.py    | 6 ++++--
 utils/augmentations.py | 3 +--
 2 files changed, 5 insertions(+), 4 deletions(-)


## 7. Start Training (BiSeNetV2 on T4 GPU)
The Colab configuration uses `batch_size: 8` and `num_workers: 4` to fully leverage the 15 GB VRAM and multi-core CPU of Google Colab.

In [ ]:
!python scripts/train.py --config configs/train_colab.yaml

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: divy32271 (divy32271-ramdeobaba-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run 81mmmkab (0.0s)
wandb: ⣻ setting up run 81mmmkab (0.0s)
wandb: ⣽ setting up run 81mmmkab (0.0s)
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/Capstone-Project/wandb/run-20261026_051901-81mmmkab
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bisenetv2
wandb: ⭐️ View project at https://wandb.ai/divy32271-ramdeobaba-university/capstone-walkability
wandb: 🚀 View run at https://wandb.ai/divy32271-ramdeobaba-university/capstone-walkability/runs/81mmmkab
[2026-10-26 05:19:03] [INFO] Loaded 2975 training samples and 500 validation samples across 1 active dataset(s).
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will creat

## 8. Backup Best Checkpoints to Google Drive
Run this to guarantee your weights are saved to your Google Drive even after the Colab session disconnects.

In [ ]:
backup_dir = "/content/drive/MyDrive/capstone_checkpoints"
os.makedirs(backup_dir, exist_ok=True)

!cp -r checkpoints/* "{backup_dir}/"
print(f"✅ All checkpoints copied to Google Drive: {backup_dir}")

## 9. Evaluate the Best Model
Compute mIoU, per-class IoU, Obstacle Recall, and Missed Obstacle Rate on the validation split.

In [ ]:
!python scripts/evaluate.py --config configs/train_colab.yaml --checkpoint checkpoints/best.pth --split val